In [ ]:
%%time
import snapatac2 as snap
from pathlib import Path

INPUT_DIR = Path("data/scATAC")
OUT_DIR   = Path("snapatac2_h5ad")
OUT_DIR.mkdir(exist_ok=True)

fragment_files = sorted(INPUT_DIR.glob("*_fragments.tsv.gz"))

out_files = [
    str(OUT_DIR / (f.stem.replace("_fragments", "") + ".h5ad"))
    for f in fragment_files
]

adatas = snap.pp.import_fragments(
    fragment_files,
    file=out_files,
    chrom_sizes=snap.genome.hg38,
    min_num_fragments=1000,
    sorted_by_barcode=False,   # ← 关键：告诉它文件未排序，让它内部排
)

print(f"\n✅ 导入完成: {len(adatas)} 个样本")
for i, adata in enumerate(adatas):
    print(f"  {fragment_files[i].stem}: {adata.n_obs} cells")


In [ ]:
import snapatac2 as snap
from pathlib import Path

In [ ]:
import snapatac2 as snap
import numpy as np

GENE_ANNO = "data/scATAC/reference/gencode.v41.basic.annotation.gff3.gz"

# ── 计算 TSS enrichment ──
snap.metrics.tsse(adatas, gene_anno=GENE_ANNO)

# ── 查看结果 ──
for i, adata in enumerate(adatas):
    ts = adata.obs['tsse']
    nf = adata.obs['n_fragment']
    print(f"--- {fragment_files[i].stem} ({adata.n_obs} cells) ---")
    print(f"  n_frag: median={np.median(nf):.0f}, mean={np.mean(nf):.0f}")
    print(f"  tsse≥10: {(ts>=10).sum()} ({(ts>=10).mean()*100:.1f}%)")
    print(f"  tsse≥5:  {(ts>=5).sum()} ({(ts>=5).mean()*100:.1f}%)")
    print()


In [ ]:
# ── 查看结果 ──
for i, adata in enumerate(adatas):
    ts = adata.obs['tsse']
    nf = adata.obs['n_fragment']
    print(f"--- {fragment_files[i].stem} ({adata.n_obs} cells) ---")

    print()

In [ ]:
import matplotlib.pyplot as plt

sample_names = [f.stem.replace('_fragments', '') for f in fragment_files]

fig, axes = plt.subplots(len(adatas), 2, figsize=(14, 4*len(adatas)))
if len(adatas) == 1:
    axes = axes.reshape(1, -1)

for i, adata in enumerate(adatas):
    short_name = sample_names[i][-20:]
    
    # Fragment count
    axes[i, 0].hist(adata.obs['n_fragment'], bins=100, color='steelblue', alpha=0.7)
    axes[i, 0].axvline(1000,  color='red', ls='--', lw=1, label='min=1000')
    axes[i, 0].axvline(50000, color='red', ls='--', lw=1, label='max=50000')
    axes[i, 0].set_xlabel('Fragments')
    axes[i, 0].set_ylabel('Cells')
    axes[i, 0].set_title(f'{short_name} — Fragments')
    axes[i, 0].legend(fontsize=7)

    # TSS enrichment
    axes[i, 1].hist(adata.obs['tsse'], bins=100, color='darkorange', alpha=0.7)
    axes[i, 1].axvline(10, color='red', ls='--', lw=1, label='tsse=10')
    axes[i, 1].set_xlabel('TSS enrichment')
    axes[i, 1].set_ylabel('Cells')
    axes[i, 1].set_title(f'{short_name} — TSSe')
    axes[i, 1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('qc_histograms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import numpy as np

for i, adata in enumerate(adatas):
    ts = adata.obs['tsse'].to_numpy()
    nf = adata.obs['n_fragment'].to_numpy()
    print(f"--- {fragment_files[i].stem} ({adata.n_obs} cells) ---")
    print(f"  tsse:  median={np.median(ts):.1f}, mean={np.mean(ts):.1f}")
    print(f"  n_frag: median={np.median(nf):.0f}, mean={np.mean(nf):.0f}")
    print(f"  tsse≥10: {(ts>=10).sum()} ({(ts>=10).mean()*100:.1f}%)")
    print(f"  tsse≥5:  {(ts>=5).sum()} ({(ts>=5).mean()*100:.1f}%)")
    print()


In [ ]:
# ===== Step 1: 看 adatas 是什么 =====
print("adatas 类型:", type(adatas))
print("adatas 长度:", len(adatas))
print("adatas[0] 类型:", type(adatas[0]))
print()

# ===== Step 2: 看 adatas[0] 的结构 =====
print(adatas[0])


In [ ]:
# ===== Step 3: 看 fragment 文件的染色体名（这个不依赖 adatas）=====
import gzip
print("fragment 文件前5行:")
with gzip.open(fragment_files[0], 'rt') as f:
    for i, line in enumerate(f):
        if i >= 5:
            break
        print(" ", line.strip())


In [ ]:
# ===== Step 4: 看 GFF3 的染色体名 =====
GENE_ANNO = "data/scATAC/reference/gencode.v41.basic.annotation.gff3.gz"
print("GFF3 前5行数据行:")
with gzip.open(GENE_ANNO, 'rt') as f:
    count = 0
    for line in f:
        if line.startswith('#'):
            continue
        print(" ", line.strip()[:80])
        count += 1
        if count >= 5:
            break


In [ ]:
# ===== Step 5: 看 snap.genome.hg38 =====
print("snap.genome.hg38 类型:", type(snap.genome.hg38))
try:
    print(snap.genome.hg38[:5])
except:
    print(snap.genome.hg38)

In [ ]:
# ===== 1. fragment 文件的数据行（跳过注释）=====
import gzip
print("fragment 数据行:")
with gzip.open(fragment_files[0], 'rt') as f:
    count = 0
    for line in f:
        if line.startswith('#'):
            continue
        print(" ", line.strip()[:80])
        count += 1
        if count >= 5:
            break


In [ ]:
# ===== 2. adata 里的染色体信息 =====
adata0 = adatas[0]
print("uns 中的 reference_sequences:")
print(adata0.uns['reference_sequences'][:5])


In [ ]:
# ===== 3. GFF3 的染色体名（跳过注释）=====
GENE_ANNO = "data/scATAC/reference/gencode.v41.basic.annotation.gff3.gz"
import gzip
print("GFF3 数据行:")
with gzip.open(GENE_ANNO, 'rt') as f:
    count = 0
    for line in f:
        if line.startswith('#'):
            continue
        fields = line.strip().split('\t')
        print(f"  chrom={fields[0]}, type={fields[2]}")
        count += 1
        if count >= 5:
            break


In [ ]:
# ===== 4. snap.genome.hg38 的染色体名 =====
hg38 = snap.genome.hg38
print("hg38 属性:", dir(hg38))
try:
    print("hg38.chromosomes:", hg38.chromosomes[:5])
except:
    pass
try:
    print("hg38.chrom_sizes:", list(hg38.chrom_sizes.items())[:5])
except:
    pass


In [ ]:
# ===== 检查1: library_tsse（整体样本质量）=====
for i, adata in enumerate(adatas):
    lib_tsse = adata.uns.get('library_tsse', 'N/A')
    print(f"  {fragment_files[i].stem}: library_tsse={lib_tsse}")


In [ ]:
# ===== 检查2: fragment_paired 是否有数据 =====
for i, adata in enumerate(adatas):
    fp = adata.obsm.get('fragment_paired', None)
    if fp is not None:
        print(f"  {fragment_files[i].stem}: fragment_paired shape={fp.shape}")
    else:
        print(f"  {fragment_files[i].stem}: fragment_paired=None ⚠️")


In [ ]:
# ===== 检查3: TSS profile（看是否有 enrichment 信号）=====
# 对比一个好样本(OR109)和一个差样本(OR101)
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for idx, label in [(2, 'OR109'), (0, 'OR101')]:
    profile = adatas[idx].uns.get('TSS_profile', None)
    if profile is not None:
        axes[idx == 0].plot(profile)
        axes[idx == 0].set_title(f'{label} - TSS profile')
        axes[idx == 0].set_xlabel('Position')
        axes[idx == 0].set_ylabel('Signal')
    else:
        axes[idx == 0].set_title(f'{label} - No TSS profile')

plt.tight_layout()
plt.show()


In [ ]:
# ── 散点图 ──
fig, ax = plt.subplots(figsize=(10, 8))
for i, adata in enumerate(adatas):
    short = sample_names[i][-15:]
    ax.scatter(adata.obs['n_fragment'], adata.obs['tsse'],
               s=1, alpha=0.15, label=short)

ax.set_xscale('log')
ax.axhline(10,  color='red', ls='--', alpha=0.5, label='tsse≥10')
ax.axvline(1000, color='red', ls='--', alpha=0.5, label='frag≥1000')
ax.set_xlabel('Fragments per cell')
ax.set_ylabel('TSS enrichment')
ax.legend(markerscale=10, fontsize=6, ncol=2)
plt.tight_layout()
plt.savefig('qc_scatter.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
import matplotlib.pyplot as plt

sample_names = [f.stem.replace('_fragments', '') for f in fragment_files]

fig, axes = plt.subplots(len(adatas), 2, figsize=(14, 4*len(adatas)))
if len(adatas) == 1:
    axes = axes.reshape(1, -1)

for i, adata in enumerate(adatas):
    short_name = sample_names[i][-20:]
    ts = adata.obs['tsse'].to_numpy()
    nf = adata.obs['n_fragment'].to_numpy()
    
    axes[i, 0].hist(nf, bins=100, color='steelblue', alpha=0.7)
    axes[i, 0].axvline(2000, color='red', ls='--', lw=1, label='min=2000')
    axes[i, 0].set_xlabel('Fragments')
    axes[i, 0].set_ylabel('Cells')
    axes[i, 0].set_title(f'{short_name} — Fragments')
    axes[i, 0].legend(fontsize=7)

    axes[i, 1].hist(ts, bins=100, color='darkorange', alpha=0.7)
    axes[i, 1].axvline(6, color='red', ls='--', lw=1, label='tsse=6')
    axes[i, 1].set_xlabel('TSS enrichment')
    axes[i, 1].set_ylabel('Cells')
    axes[i, 1].set_title(f'{short_name} — TSSe')
    axes[i, 1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('qc_v2_histograms.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ===== NC 论文参数 =====
MIN_COUNTS = 2000
MAX_COUNTS = 100000
MIN_TSSE   = 6

print("===== 过滤前 =====")
for i, adata in enumerate(adatas):
    print(f"  {sample_names[i]}: {adata.n_obs} cells")

snap.pp.filter_cells(
    adatas,
    min_counts=MIN_COUNTS,
    max_counts=MAX_COUNTS,
    min_tsse=MIN_TSSE,
)

snap.pp.filter_features(
    adatas,
    min_count=10,
)

print("\n===== 过滤后 =====")
for i, adata in enumerate(adatas):
    print(f"  {sample_names[i]}: {adata.n_obs} cells")

total = sum(a.n_obs for a in adatas)
print(f"\n总计: {total} cells")


In [ ]:
print("\n===== 过滤后 =====")
for i, adata in enumerate(adatas):
    print(f"  {sample_names[i]}: {adata.n_obs} cells")

total = sum(a.n_obs for a in adatas)
print(f"\n总计: {total} cells")

In [ ]:
%%time
snap.pp.add_tile_matrix(adatas, bin_size=5000)
snap.pp.select_features(adatas, n_features=50000)
snap.pp.scrublet(adatas)
snap.pp.filter_doublets(adatas)

In [ ]:
%%time
from pathlib import Path
import snapatac2 as snap

OUT_DIR = Path("data/scATAC")
OUT_DIR.mkdir(exist_ok=True)


# 合并为 AnnDataSet（磁盘模式，不占内存）
data = snap.AnnDataSet(
    adatas=[(sample_names[i], adatas[i]) for i in range(len(adatas))],
    filename=str(OUT_DIR / "combind.h5ads"),
)



In [ ]:
import snapatac2 as snap
from pathlib import Path

OUT_DIR = Path("data/scATAC")

# 用 snap.read 打开已保存的文件
data = snap.read(str(OUT_DIR / "combind.h5ads"), backed="r+")

print(f"✅ 加载完成: {data.n_obs} cells × {data.n_vars} features")


In [ ]:
import snapatac2 as snap
from pathlib import Path
import polars as pl

OUT_DIR = Path("data/scATAC")
H5AD_DIR = OUT_DIR / "snapatac2_h5ad"

# 1. 找到所有 h5ad 文件
h5ad_files = sorted(H5AD_DIR.glob("*.h5ad"))
print(f"找到 {len(h5ad_files)} 个 h5ad 文件:")
for f in h5ad_files:
    print(f"  {f.name}")

# 2. 加载每个 h5ad
adatas = []
sample_names = []
for f in h5ad_files:
    name = f.stem
    sample_names.append(name)
    adata = snap.read(str(f), backed="r+")
    adatas.append(adata)
    print(f"  {name}: {adata.n_obs} cells")


In [ ]:
import os
merged_path = str(OUT_DIR / "combind.h5ads")
if os.path.exists(merged_path):
    os.remove(merged_path)

# 直接合并，不修改单个 adata
data = snap.AnnDataSet(
    adatas=[(sample_names[i], adatas[i]) for i in range(len(adatas))],
    filename=merged_path,
)

print(f"✅ 合并完成: {data.n_obs} cells × {data.n_vars} features")


In [ ]:
# 构建 condition 标签
import numpy as np
samples_arr = np.array(data.obs['sample'])
group_map = {s: ('Control' if 'Control' in s else 'UC') for s in set(samples_arr)}
data.obs['condition'] = np.array([group_map[s] for s in samples_arr])

print("Group 映射:")
for s, g in sorted(group_map.items()):
    print(f"  {s} → {g}")

print(f"\n总计: UC={sum(data.obs['condition']=='UC'):,}  |  Control={sum(data.obs['condition']=='Control'):,}")

In [ ]:
snap.pp.select_features(data, n_features=50000)

In [ ]:
snap.tl.spectral(data, n_comps=50)

In [ ]:
print(f"✅ X_spectral: {data.obsm['X_spectral'].shape}")

In [ ]:
%%time
snap.tl.umap(data)

In [ ]:
data.close()
data

In [ ]:

data = snap.read_dataset("data/scATAC/combind.h5ads")
data

In [ ]:
snap.pl.umap(data, color="sample", interactive=True)

In [ ]:
%%time
snap.pp.mnc_correct(data, batch="sample")

In [ ]:
data

In [ ]:
import pandas as pd
import scanpy as sc
import numpy as np

# 手工构建临时 AnnData（绕过 AnnDataSet 的 obs 限制）
samples_list = list(data.obs['sample'])
obs_df = pd.DataFrame({
    'sample': samples_list,
    'condition': np.where(['Control' in s for s in samples_list], 'Control', 'UC')
}, index=[str(i) for i in range(len(samples_list))])

tmp = sc.AnnData(X=data.obsm["X_spectral"], obs=obs_df)
tmp.obsm["X_pca"] = data.obsm["X_spectral"]

# BBKNN 批次校正
sc.pp.neighbors(tmp, n_pcs=50)
import bbknn
bbknn.bbknn(tmp, batch_key='sample')
sc.tl.umap(tmp)

# 回写给 SnapATAC2
data.obsm["X_umap"] = tmp.obsm["X_umap"]
use_rep = "X_spectral"
print("✅ BBKNN 批次校正完成")

In [ ]:
data.obsm["X_umap"] = tmp.obsm["X_umap"]

In [ ]:
snap.pl.umap(data, color="sample", interactive=True)

In [ ]:
snap.pp.knn(data)

In [ ]:

snap.tl.leiden(data)



In [ ]:

snap.pl.umap(data, color="leiden", interactive=True)

In [ ]:
%%time
snap.tl.macs3(data, groupby='leiden', replicate='sample')

In [ ]:
merged_peaks = snap.tl.merge_peaks(data.uns['macs3'], chrom_sizes=snap.genome.hg38)
merged_peaks.head()

In [ ]:
data.close()
data

In [ ]:
merged_peaks.head()

In [ ]:
data = snap.read_dataset("data/scATAC/combind.h5ads")
data

In [ ]:
snap.pl.frag_size_distr(data, interactive=False)

In [ ]:
snap.pl.coverage(data, region='chr16:28930971-28932971', groupby='leiden')